## Example 1: Basic Code Linting and Style Checking

In [ ]:
from flake8.api import legacy as flake8
import tempfile
import os

# Example 1: Basic code linting and style checking
print("Example 1: Basic Code Linting and Style Checking")

# Create sample Python code with various issues
sample_code = """
import os,sys
import json

def bad_function( x,y ):
    if x>5:
        result=x+y
        unused_var = 123
        print("Hello world")
    return result

class   MyClass:
    def method1(self):
        pass
        
def another_function():
    import time  # Import not at top
    if True:
        x = 1
        y = 2
        z = x+y
    return z
"""

# Create temporary file with the code
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as temp_file:
    temp_file.write(sample_code)
    temp_path = temp_file.name

try:
    # Create flake8 style guide instance. flake8's stock reporter writes to `sys.stdout.buffer`, which a
    # Jupyter kernel's stdout does not have; the legacy API's `init_report` takes a formatter class, and
    # this one keeps each formatted violation instead of writing it.
    from flake8.formatting.default import Default

    class CollectingReport(Default):
        lines = []

        def write(self, line, source):
            CollectingReport.lines.append(line)

    style_guide = flake8.get_style_guide()
    style_guide.init_report(CollectingReport)
    
    # Check the file for violations
    report = style_guide.check_files([temp_path])
    
    print(f"Total violations found: {report.total_errors}")
    print(f"First formatted violation: {CollectingReport.lines[0]}")
    assert report.total_errors == len(CollectingReport.lines) > 0, (report.total_errors, CollectingReport.lines[:3])
    
    # Show statistics
    stats = report.get_statistics('E')  # Error statistics
    print(f"\nError statistics:")
    for stat in stats[:5]:  # Show first 5 error types
        print(f"  {stat}")
        
    # Check for specific violation types
    print(f"\nChecking code quality:")
    print(f"File checked: {os.path.basename(temp_path)}")
    print(f"Lines of code analyzed: {len(sample_code.splitlines())}")
    
finally:
    # Clean up temporary file
    os.unlink(temp_path)

print("Basic linting completed successfully!")

## Example 2: Configuration and Custom Rules

In [ ]:
from flake8.api import legacy as flake8
import tempfile
import os

# Example 2: Configuration and custom rules
print("Example 2: Configuration and Custom Rules")

# Sample code with long lines and complexity
complex_code = """
def complex_function(param1, param2, param3, param4, param5, param6, param7, param8):
    # This line is intentionally very long to demonstrate line length checking and how flake8 handles it
    if param1 and param2 and param3 and param4 and param5 and param6 and param7 and param8:
        if param1 > 10:
            if param2 < 5:
                if param3 == "test":
                    if param4 is not None:
                        if param5 > param6:
                            if param7 != param8:
                                return "Very nested result"
                            else:
                                return "Alternative result"
                        else:
                            return "Another alternative"
                    else:
                        return "Param4 is None"
                else:
                    return "Not test"
            else:
                return "Param2 not less than 5"
        else:
            return "Param1 not greater than 10"
    else:
        return "Some param is falsy"
"""

# Create temporary file
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as temp_file:
    temp_file.write(complex_code)
    temp_path = temp_file.name

try:
    # Configure flake8 with custom settings
    print("Checking with default configuration:")
    style_guide_default = flake8.get_style_guide(quiet=2)  # quiet=2: the stock reporter writes to sys.stdout.buffer, which Jupyter's stdout lacks
    report_default = style_guide_default.check_files([temp_path])
    print(f"Default violations: {report_default.total_errors}")
    
    # Configure with custom line length and complexity
    print("\nChecking with custom configuration (shorter lines, lower complexity):")
    style_guide_custom = flake8.get_style_guide(
        quiet=2,  # quiet=2: the stock reporter writes to sys.stdout.buffer, which Jupyter's stdout lacks
        max_line_length=80,
        max_complexity=5,
        ignore=['E302', 'E305']  # Ignore some spacing rules
    )
    report_custom = style_guide_custom.check_files([temp_path])
    print(f"Custom config violations: {report_custom.total_errors}")
    
    # Show specific error types
    error_stats = report_custom.get_statistics('E')
    warning_stats = report_custom.get_statistics('W')
    complexity_stats = report_custom.get_statistics('C')
    
    print(f"\nDetailed statistics:")
    print(f"Errors (E): {len(error_stats)}")
    print(f"Warnings (W): {len(warning_stats)}")  
    print(f"Complexity (C): {len(complexity_stats)}")
    
    if complexity_stats:
        print("Complexity violations:")
        for stat in complexity_stats[:3]:
            print(f"  {stat}")
    
finally:
    os.unlink(temp_path)

print("\nConfiguration testing completed successfully!")

## Example 3: Multiple File Analysis and Reporting

In [ ]:
from flake8.api import legacy as flake8
import tempfile
import os

# Example 3: Multiple file analysis and reporting
print("Example 3: Multiple File Analysis and Reporting")

# Create multiple Python files with different issue types
files_data = {
    'module1.py': """
import os
import sys

def function1():
    x = 1
    y = 2
    return x + y

class MyClass:
    def __init__(self):
        self.value = 42
""",
    'module2.py': """
from os import path,getcwd
import json

def problematic_function(a,b,c):
    if a>b:
        result=a+b+c
        unused_variable = "test"
        print( "Debug message" )
    return result

def another_func():
    pass
""",
    'module3.py': """
import sys

def good_function(param1, param2):
    \"\"\"This is a well-formatted function.\"\"\"
    if param1 is not None and param2 > 0:
        result = param1 * param2
        return result
    return 0

class WellFormattedClass:
    \"\"\"A properly formatted class.\"\"\"
    
    def __init__(self, value):
        self.value = value
    
    def get_value(self):
        return self.value
"""
}

# Create temporary directory and files
temp_dir = tempfile.mkdtemp()
temp_files = []

try:
    for filename, content in files_data.items():
        file_path = os.path.join(temp_dir, filename)
        with open(file_path, 'w') as f:
            f.write(content)
        temp_files.append(file_path)
    
    # Run flake8 on all files
    style_guide = flake8.get_style_guide(
        quiet=2,  # quiet=2: the stock reporter writes to sys.stdout.buffer, which Jupyter's stdout lacks
        statistics=True,
        count=True,
        max_line_length=88
    )
    
    print(f"Analyzing {len(temp_files)} Python files...")
    report = style_guide.check_files(temp_files)
    
    print(f"Total violations across all files: {report.total_errors}")
    
    # Get detailed statistics
    all_stats = []
    for error_code in ['E', 'W', 'F']:
        stats = report.get_statistics(error_code)
        all_stats.extend(stats)
    
    print(f"\nTop violation types:")
    for i, stat in enumerate(all_stats[:8]):
        print(f"  {i+1}. {stat}")
    
    # Analyze by file
    print(f"\nPer-file analysis:")
    for i, file_path in enumerate(temp_files, 1):
        filename = os.path.basename(file_path)
        print(f"  File {i}: {filename}")
        
        # Count lines in each file
        with open(file_path, 'r') as f:
            line_count = len(f.readlines())
        print(f"    Lines of code: {line_count}")
    
    print(f"\nMulti-file analysis completed successfully!")
    
finally:
    # Clean up temporary files and directory
    for file_path in temp_files:
        if os.path.exists(file_path):
            os.unlink(file_path)
    os.rmdir(temp_dir)

## Example 4: Error Classification and Code Quality Metrics

In [ ]:
from flake8.api import legacy as flake8
import tempfile
import os

# Example 4: Error classification and code quality metrics  
print("Example 4: Error Classification and Code Quality Metrics")

# Create code samples with different types of violations
violation_samples = {
    'syntax_issues.py': """
# Syntax and import issues
import os, sys, json
from collections import defaultdict,Counter

def func_with_issues():
    x=1+2  # Missing spaces around operator
    y = 3+   4  # Inconsistent spacing
    unused_var = "never used"
    return x,y  # Missing space after comma
""",
    'style_issues.py': """
# PEP8 style violations


def   function_with_bad_spacing  ():
    pass




class BadClass:
    def __init__( self ):
        pass
    def method1(self):pass
    def method2(self) :
        pass
""",
    'complexity_issues.py': """
# McCabe complexity violations
def overly_complex_function(a, b, c, d, e):
    if a > 0:
        if b < 10:
            if c == "test":
                if d is not None:
                    if e > 5:
                        if a + b > c:
                            if d != e:
                                if len(str(a)) > 2:
                                    if b % 2 == 0:
                                        if c.startswith("t"):
                                            return "deeply nested result"
    return "default"
""",
    'clean_code.py': """
# Well-formatted code with minimal violations
import os
import sys
from typing import Optional


def clean_function(param1: int, param2: str) -> Optional[str]:
    \"\"\"A well-formatted function with proper typing.\"\"\"
    if param1 > 0 and param2:
        result = f"Processed: {param1} - {param2}"
        return result
    return None


class CleanClass:
    \"\"\"A properly formatted class.\"\"\"
    
    def __init__(self, value: int) -> None:
        self.value = value
    
    def process(self) -> int:
        return self.value * 2
"""
}

# Create temporary directory and analyze each file type
temp_dir = tempfile.mkdtemp()

try:
    violation_counts = {}
    
    for filename, code in violation_samples.items():
        file_path = os.path.join(temp_dir, filename)
        with open(file_path, 'w') as f:
            f.write(code)
        
        # Analyze each file individually
        style_guide = flake8.get_style_guide(quiet=2)  # quiet=2: the stock reporter writes to sys.stdout.buffer, which Jupyter's stdout lacks
        report = style_guide.check_files([file_path])
        
        violation_counts[filename] = report.total_errors
        
        print(f"\n{filename}:")
        print(f"  Total violations: {report.total_errors}")
        
        # Get statistics by error type
        for error_type in ['E', 'W', 'F', 'C']:
            stats = report.get_statistics(error_type)
            if stats:
                print(f"  {error_type}-type violations: {len(stats)}")
                if len(stats) <= 3:  # Show details for few violations
                    for stat in stats:
                        print(f"    {stat}")
    
    # Summary analysis
    print(f"\n{'='*50}")
    print("Code Quality Summary:")
    print(f"{'='*50}")
    
    sorted_files = sorted(violation_counts.items(), key=lambda x: x[1])
    
    print("Files ranked by code quality (fewest violations first):")
    for i, (filename, count) in enumerate(sorted_files, 1):
        quality_rating = "Excellent" if count <= 2 else "Good" if count <= 5 else "Needs Work" if count <= 10 else "Poor"
        print(f"  {i}. {filename}: {count} violations - {quality_rating}")
    
    total_violations = sum(violation_counts.values())
    avg_violations = total_violations / len(violation_counts)
    
    print(f"\nOverall Project Metrics:")
    print(f"  Total violations: {total_violations}")
    print(f"  Average per file: {avg_violations:.1f}")
    print(f"  Files analyzed: {len(violation_counts)}")
    
finally:
    # Clean up
    for filename in violation_samples.keys():
        file_path = os.path.join(temp_dir, filename)
        if os.path.exists(file_path):
            os.unlink(file_path)
    os.rmdir(temp_dir)

print("\nError classification analysis completed successfully!")

## Example 5: Integration with Development Workflow and Automation

In [ ]:
from flake8.api import legacy as flake8
import tempfile
import os
import json

# Example 5: Integration with development workflow and automation
print("Example 5: Integration with Development Workflow and Automation")

# Simulate a development project structure
project_files = {
    'src/main.py': """
import os
from typing import List, Dict

def process_data(data: List[Dict]) -> List[Dict]:
    \"\"\"Process input data and return results.\"\"\"
    results = []
    for item in data:
        if 'name' in item and 'value' in item:
            processed = {
                'name': item['name'].strip(),
                'value': item['value'] * 2,
                'processed': True
            }
            results.append(processed)
    return results
""",
    'src/utils.py': """
# Some utility functions with minor issues
import sys,os
import json

def helper_func(x,y):
    result=x+y
    unused_var = "test"
    return result

def another_helper():
    pass
""",
    'tests/test_main.py': """
import unittest
from src.main import process_data

class TestMain(unittest.TestCase):
    def test_process_data(self):
        input_data = [{'name': '  test  ', 'value': 5}]
        result = process_data(input_data)
        
        expected = [{'name': 'test', 'value': 10, 'processed': True}]
        self.assertEqual(result, expected)
        
    def test_empty_data(self):
        result = process_data([])
        self.assertEqual(result, [])
"""
}

# Create project structure
temp_dir = tempfile.mkdtemp()

try:
    # Create directories and files
    for file_path, content in project_files.items():
        full_path = os.path.join(temp_dir, file_path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, 'w') as f:
            f.write(content)
    
    print("Created mock project structure:")
    for file_path in project_files.keys():
        print(f"  {file_path}")
    
    # Run comprehensive project analysis
    all_py_files = []
    for root, dirs, files in os.walk(temp_dir):
        for file in files:
            if file.endswith('.py'):
                all_py_files.append(os.path.join(root, file))
    
    print(f"\nAnalyzing {len(all_py_files)} Python files...")
    
    # Configure for CI/CD-like environment
    style_guide = flake8.get_style_guide(
        quiet=2,  # quiet=2: the stock reporter writes to sys.stdout.buffer, which Jupyter's stdout lacks
        max_line_length=88,
        statistics=True,
        count=True,
        max_complexity=10
    )
    
    report = style_guide.check_files(all_py_files)
    
    # Generate CI/CD report
    ci_report = {
        'total_files': len(all_py_files),
        'total_violations': report.total_errors,
        'violations_per_file': report.total_errors / len(all_py_files),
        'status': 'PASS' if report.total_errors <= 5 else 'FAIL',
        'details': {}
    }
    
    # Get violation breakdown
    for error_type in ['E', 'W', 'F', 'C']:
        stats = report.get_statistics(error_type)
        ci_report['details'][f'{error_type}_violations'] = len(stats)
    
    print(f"\nCI/CD Quality Report:")
    print(f"{'='*40}")
    print(f"Status: {ci_report['status']}")
    print(f"Total Files: {ci_report['total_files']}")
    print(f"Total Violations: {ci_report['total_violations']}")
    print(f"Avg Violations/File: {ci_report['violations_per_file']:.1f}")
    
    print(f"\nViolation Breakdown:")
    for key, value in ci_report['details'].items():
        if value > 0:
            print(f"  {key}: {value}")
    
    # Simulate pre-commit hook behavior
    print(f"\nPre-commit Hook Simulation:")
    if ci_report['total_violations'] == 0:
        print("✅ All checks passed - ready to commit")
    elif ci_report['total_violations'] <= 3:
        print("⚠️  Minor violations found - review recommended")
    else:
        print("❌ Code quality issues found - fix before commit")
    
    # Generate quality metrics for tracking
    quality_score = max(0, 100 - (ci_report['total_violations'] * 5))
    print(f"\nCode Quality Score: {quality_score}/100")
    
    # Suggest automated fixes
    if ci_report['total_violations'] > 0:
        print(f"\nAutomation Suggestions:")
        print("  - Run 'autopep8' for automatic PEP8 fixes")
        print("  - Use 'isort' for import organization")
        print("  - Consider 'black' for code formatting")
        print("  - Add flake8 to pre-commit hooks")
    
finally:
    # Clean up project structure
    for root, dirs, files in os.walk(temp_dir, topdown=False):
        for file in files:
            os.unlink(os.path.join(root, file))
        for dir in dirs:
            os.rmdir(os.path.join(root, dir))
    os.rmdir(temp_dir)

print("\nWorkflow integration analysis completed successfully!")